# PHẦN II — NETWORK FABRIC (VXLAN OVERLAY)

## Mục tiêu phần này

Trong mô hình này, mỗi server vật lý chỉ có **1 NIC vật lý chính** đi ra ngoài qua `br-lab`.

Các mạng nội bộ phục vụ OpenStack sẽ được tạo bằng **VXLAN overlay** chạy trên nền `br-lab`.

| Bridge | Mục đích | VXLAN ID | IP Server A | IP Server B | IP Server C |
|---|---|---:|---|---|---|
| `br-mgmt` | Management Network | 100 | `10.0.0.21/24` | `10.0.0.22/24` | `10.0.0.23/24` |
| `br-storage` | Storage Network | 200 | `10.0.2.21/24` | `10.0.2.22/24` | `10.0.2.23/24` |
| `br-cluster` | Ceph Replication / Cluster Network | 300 | `10.0.3.21/24` | `10.0.3.22/24` | `10.0.3.23/24` |
| `br-ex` | Provider / Floating IP NAT | - | không cần IP | không cần IP | không cần IP |

## Mapping server vật lý

- Server A: `192.168.150.9`
- Server B: `192.168.150.10`
- Server C: `192.168.150.11`

## Nguyên tắc quan trọng

- Chạy đúng block theo đúng server.
- Không copy block Server A sang Server B/C nếu chưa đổi `local`, `remote`, IP bridge.
- VXLAN dùng UDP port `4789`.
- MTU nên để `1450` vì VXLAN có overhead.
- `br-ex` nối với `br-lab` bằng `veth pair` để Floating IP NAT ra ngoài.


## BƯỚC 7 — Tạo `br-mgmt` Management Network

`br-mgmt` là mạng quản trị nội bộ giữa 3 server.

Mạng này dùng cho:
- Controller VM / service management
- SSH nội bộ nếu cần
- API nội bộ OpenStack
- traffic quản trị giữa các node

VXLAN ID sử dụng: `100`


### BƯỚC 7.1 — Server A tạo `br-mgmt`

Chạy block này **chỉ trên Server A**: `192.168.150.9`

Kết quả mong muốn:
- `br-mgmt` có IP `10.0.0.21/24`
- Có 2 VXLAN tunnel:
  - đến Server B: `192.168.150.10`
  - đến Server C: `192.168.150.11`


In [ ]:
# ============================================================
# BƯỚC 7.1 — SERVER A: tạo VXLAN id=100 và bridge br-mgmt
# Server A underlay IP: 192.168.150.9
# Server B underlay IP: 192.168.150.10
# Server C underlay IP: 192.168.150.11
# br-mgmt IP của Server A: 10.0.0.21/24
# ============================================================

# Tạo VXLAN tunnel từ Server A đến Server B
sudo ip link add vxlan100-b type vxlan id 100 \
  local 192.168.150.9 remote 192.168.150.10 dstport 4789 nolearning

# Tạo VXLAN tunnel từ Server A đến Server C
sudo ip link add vxlan100-c type vxlan id 100 \
  local 192.168.150.9 remote 192.168.150.11 dstport 4789 nolearning

# Tạo Linux bridge br-mgmt
sudo ip link add br-mgmt type bridge

# Gắn 2 interface VXLAN vào br-mgmt
sudo ip link set vxlan100-b master br-mgmt
sudo ip link set vxlan100-c master br-mgmt

# Gán IP management cho br-mgmt trên Server A
sudo ip addr add 10.0.0.21/24 dev br-mgmt

# Bật bridge và các VXLAN interface
sudo ip link set br-mgmt up
sudo ip link set vxlan100-b up
sudo ip link set vxlan100-c up

# Khuyến nghị set MTU 1450 cho overlay VXLAN
sudo ip link set br-mgmt mtu 1450
sudo ip link set vxlan100-b mtu 1450
sudo ip link set vxlan100-c mtu 1450

# Xác nhận br-mgmt đã có IP đúng
ip addr show br-mgmt

# Xác nhận VXLAN interface đã up
ip link show vxlan100-b
ip link show vxlan100-c


### BƯỚC 7.2 — Server B tạo `br-mgmt`

Chạy block này **chỉ trên Server B**: `192.168.150.10`

Kết quả mong muốn:
- `br-mgmt` có IP `10.0.0.22/24`
- Có tunnel đến Server A và Server C


In [ ]:
# ============================================================
# BƯỚC 7.2 — SERVER B: tạo VXLAN id=100 và bridge br-mgmt
# Server B underlay IP: 192.168.150.10
# Server A underlay IP: 192.168.150.9
# Server C underlay IP: 192.168.150.11
# br-mgmt IP của Server B: 10.0.0.22/24
# ============================================================

# Tạo VXLAN tunnel từ Server B đến Server A
sudo ip link add vxlan100-a type vxlan id 100 \
  local 192.168.150.10 remote 192.168.150.9 dstport 4789 nolearning

# Tạo VXLAN tunnel từ Server B đến Server C
sudo ip link add vxlan100-c type vxlan id 100 \
  local 192.168.150.10 remote 192.168.150.11 dstport 4789 nolearning

# Tạo bridge br-mgmt
sudo ip link add br-mgmt type bridge

# Gắn VXLAN interface vào bridge
sudo ip link set vxlan100-a master br-mgmt
sudo ip link set vxlan100-c master br-mgmt

# Gán IP cho br-mgmt trên Server B
sudo ip addr add 10.0.0.22/24 dev br-mgmt

# Bật interface
sudo ip link set br-mgmt up
sudo ip link set vxlan100-a up
sudo ip link set vxlan100-c up

# Set MTU 1450 cho overlay
sudo ip link set br-mgmt mtu 1450
sudo ip link set vxlan100-a mtu 1450
sudo ip link set vxlan100-c mtu 1450

# Verify
ip addr show br-mgmt
ip link show vxlan100-a
ip link show vxlan100-c


### BƯỚC 7.3 — Server C tạo `br-mgmt`

Chạy block này **chỉ trên Server C**: `192.168.150.11`

Kết quả mong muốn:
- `br-mgmt` có IP `10.0.0.23/24`
- Có tunnel đến Server A và Server B


In [ ]:
# ============================================================
# BƯỚC 7.3 — SERVER C: tạo VXLAN id=100 và bridge br-mgmt
# Server C underlay IP: 192.168.150.11
# Server A underlay IP: 192.168.150.9
# Server B underlay IP: 192.168.150.10
# br-mgmt IP của Server C: 10.0.0.23/24
# ============================================================

# Tạo VXLAN tunnel từ Server C đến Server A
sudo ip link add vxlan100-a type vxlan id 100 \
  local 192.168.150.11 remote 192.168.150.9 dstport 4789 nolearning

# Tạo VXLAN tunnel từ Server C đến Server B
sudo ip link add vxlan100-b type vxlan id 100 \
  local 192.168.150.11 remote 192.168.150.10 dstport 4789 nolearning

# Tạo bridge br-mgmt
sudo ip link add br-mgmt type bridge

# Gắn VXLAN vào bridge
sudo ip link set vxlan100-a master br-mgmt
sudo ip link set vxlan100-b master br-mgmt

# Gán IP br-mgmt cho Server C
sudo ip addr add 10.0.0.23/24 dev br-mgmt

# Bật interface
sudo ip link set br-mgmt up
sudo ip link set vxlan100-a up
sudo ip link set vxlan100-b up

# Set MTU 1450 cho overlay
sudo ip link set br-mgmt mtu 1450
sudo ip link set vxlan100-a mtu 1450
sudo ip link set vxlan100-b mtu 1450

# Verify
ip addr show br-mgmt
ip link show vxlan100-a
ip link show vxlan100-b


### BƯỚC 7.4 — Kiểm tra kết nối `br-mgmt`

Sau khi tạo xong `br-mgmt` trên cả 3 server, kiểm tra ping overlay.

Chạy trên Server A:


In [ ]:
# ============================================================
# TEST br-mgmt từ Server A
# ============================================================

# Ping br-mgmt của Server B
ping -c 3 -I br-mgmt 10.0.0.22

# Ping br-mgmt của Server C
ping -c 3 -I br-mgmt 10.0.0.23

# Nếu không ping được:
# 1. Kiểm tra VXLAN interface đã UP chưa: ip link | grep vxlan100
# 2. Kiểm tra br-mgmt có IP chưa: ip addr show br-mgmt
# 3. Kiểm tra firewall có chặn UDP 4789 không
# 4. Kiểm tra underlay 192.168.150.x ping nhau được không


## BƯỚC 8 — Tạo `br-storage` Storage Network

`br-storage` là mạng storage nội bộ giữa các node.

Dùng cho:
- traffic Cinder / Glance / Nova tới backend storage
- traffic RBD nếu OpenStack dùng Ceph backend
- tách storage traffic khỏi management traffic

VXLAN ID sử dụng: `200`

IP plan:
- Server A: `10.0.2.21/24`
- Server B: `10.0.2.22/24`
- Server C: `10.0.2.23/24`


### BƯỚC 8.1 — Server A tạo `br-storage`

In [ ]:
# ============================================================
# BƯỚC 8.1 — SERVER A: tạo VXLAN id=200 và bridge br-storage
# br-storage IP của Server A: 10.0.2.21/24
# ============================================================

# Tạo VXLAN tunnel tới Server B
sudo ip link add vxlan200-b type vxlan id 200 \
  local 192.168.150.9 remote 192.168.150.10 dstport 4789 nolearning

# Tạo VXLAN tunnel tới Server C
sudo ip link add vxlan200-c type vxlan id 200 \
  local 192.168.150.9 remote 192.168.150.11 dstport 4789 nolearning

# Tạo bridge storage
sudo ip link add br-storage type bridge

# Gắn VXLAN vào bridge
sudo ip link set vxlan200-b master br-storage
sudo ip link set vxlan200-c master br-storage

# Gán IP storage
sudo ip addr add 10.0.2.21/24 dev br-storage

# Bật interface
sudo ip link set br-storage up
sudo ip link set vxlan200-b up
sudo ip link set vxlan200-c up

# Set MTU 1450 do VXLAN overhead
sudo ip link set br-storage mtu 1450
sudo ip link set vxlan200-b mtu 1450
sudo ip link set vxlan200-c mtu 1450

# Verify
ip addr show br-storage
ip link show vxlan200-b
ip link show vxlan200-c


### BƯỚC 8.2 — Server B tạo `br-storage`

In [ ]:
# ============================================================
# BƯỚC 8.2 — SERVER B: tạo VXLAN id=200 và bridge br-storage
# br-storage IP của Server B: 10.0.2.22/24
# ============================================================

sudo ip link add vxlan200-a type vxlan id 200 \
  local 192.168.150.10 remote 192.168.150.9 dstport 4789 nolearning

sudo ip link add vxlan200-c type vxlan id 200 \
  local 192.168.150.10 remote 192.168.150.11 dstport 4789 nolearning

sudo ip link add br-storage type bridge
sudo ip link set vxlan200-a master br-storage
sudo ip link set vxlan200-c master br-storage

sudo ip addr add 10.0.2.22/24 dev br-storage

sudo ip link set br-storage up
sudo ip link set vxlan200-a up
sudo ip link set vxlan200-c up

sudo ip link set br-storage mtu 1450
sudo ip link set vxlan200-a mtu 1450
sudo ip link set vxlan200-c mtu 1450

ip addr show br-storage


### BƯỚC 8.3 — Server C tạo `br-storage`

In [ ]:
# ============================================================
# BƯỚC 8.3 — SERVER C: tạo VXLAN id=200 và bridge br-storage
# br-storage IP của Server C: 10.0.2.23/24
# ============================================================

sudo ip link add vxlan200-a type vxlan id 200 \
  local 192.168.150.11 remote 192.168.150.9 dstport 4789 nolearning

sudo ip link add vxlan200-b type vxlan id 200 \
  local 192.168.150.11 remote 192.168.150.10 dstport 4789 nolearning

sudo ip link add br-storage type bridge
sudo ip link set vxlan200-a master br-storage
sudo ip link set vxlan200-b master br-storage

sudo ip addr add 10.0.2.23/24 dev br-storage

sudo ip link set br-storage up
sudo ip link set vxlan200-a up
sudo ip link set vxlan200-b up

sudo ip link set br-storage mtu 1450
sudo ip link set vxlan200-a mtu 1450
sudo ip link set vxlan200-b mtu 1450

ip addr show br-storage


### BƯỚC 8.4 — Kiểm tra `br-storage`

Chạy trên Server A:


In [ ]:
# TEST br-storage từ Server A
ping -c 3 -I br-storage 10.0.2.22
ping -c 3 -I br-storage 10.0.2.23

# Kiểm tra MTU
ip link show br-storage


## BƯỚC 9 — Tạo `br-cluster` cho Ceph Replication

`br-cluster` là mạng cluster/replication của Ceph.

Dùng cho:
- OSD replication
- backfill / recovery
- heartbeat cluster nếu cấu hình Ceph dùng network này

VXLAN ID sử dụng: `300`

IP plan:
- Server A: `10.0.3.21/24`
- Server B: `10.0.3.22/24`
- Server C: `10.0.3.23/24`


### BƯỚC 9.1 — Server A tạo `br-cluster`

In [ ]:
# ============================================================
# BƯỚC 9.1 — SERVER A: tạo VXLAN id=300 và bridge br-cluster
# br-cluster IP của Server A: 10.0.3.21/24
# ============================================================

sudo ip link add vxlan300-b type vxlan id 300 \
  local 192.168.150.9 remote 192.168.150.10 dstport 4789 nolearning

sudo ip link add vxlan300-c type vxlan id 300 \
  local 192.168.150.9 remote 192.168.150.11 dstport 4789 nolearning

sudo ip link add br-cluster type bridge
sudo ip link set vxlan300-b master br-cluster
sudo ip link set vxlan300-c master br-cluster

sudo ip addr add 10.0.3.21/24 dev br-cluster

sudo ip link set br-cluster up
sudo ip link set vxlan300-b up
sudo ip link set vxlan300-c up

sudo ip link set br-cluster mtu 1450
sudo ip link set vxlan300-b mtu 1450
sudo ip link set vxlan300-c mtu 1450

ip addr show br-cluster


### BƯỚC 9.2 — Server B tạo `br-cluster`

In [ ]:
# ============================================================
# BƯỚC 9.2 — SERVER B: tạo VXLAN id=300 và bridge br-cluster
# br-cluster IP của Server B: 10.0.3.22/24
# ============================================================

sudo ip link add vxlan300-a type vxlan id 300 \
  local 192.168.150.10 remote 192.168.150.9 dstport 4789 nolearning

sudo ip link add vxlan300-c type vxlan id 300 \
  local 192.168.150.10 remote 192.168.150.11 dstport 4789 nolearning

sudo ip link add br-cluster type bridge
sudo ip link set vxlan300-a master br-cluster
sudo ip link set vxlan300-c master br-cluster

sudo ip addr add 10.0.3.22/24 dev br-cluster

sudo ip link set br-cluster up
sudo ip link set vxlan300-a up
sudo ip link set vxlan300-c up

sudo ip link set br-cluster mtu 1450
sudo ip link set vxlan300-a mtu 1450
sudo ip link set vxlan300-c mtu 1450

ip addr show br-cluster


### BƯỚC 9.3 — Server C tạo `br-cluster`

In [ ]:
# ============================================================
# BƯỚC 9.3 — SERVER C: tạo VXLAN id=300 và bridge br-cluster
# br-cluster IP của Server C: 10.0.3.23/24
# ============================================================

sudo ip link add vxlan300-a type vxlan id 300 \
  local 192.168.150.11 remote 192.168.150.9 dstport 4789 nolearning

sudo ip link add vxlan300-b type vxlan id 300 \
  local 192.168.150.11 remote 192.168.150.10 dstport 4789 nolearning

sudo ip link add br-cluster type bridge
sudo ip link set vxlan300-a master br-cluster
sudo ip link set vxlan300-b master br-cluster

sudo ip addr add 10.0.3.23/24 dev br-cluster

sudo ip link set br-cluster up
sudo ip link set vxlan300-a up
sudo ip link set vxlan300-b up

sudo ip link set br-cluster mtu 1450
sudo ip link set vxlan300-a mtu 1450
sudo ip link set vxlan300-b mtu 1450

ip addr show br-cluster


### BƯỚC 9.4 — Kiểm tra `br-cluster`

Chạy trên Server A:


In [ ]:
# TEST br-cluster từ Server A
ping -c 3 -I br-cluster 10.0.3.22
ping -c 3 -I br-cluster 10.0.3.23

# Kiểm tra MTU
ip link show br-cluster


## BƯỚC 10 — Tạo `br-ex` Provider / Floating IP

`br-ex` là bridge dành cho mạng provider / external của OpenStack.

Trong lab này chỉ có `br-lab` là bridge có uplink thật ra ngoài, nên `br-ex` sẽ kết nối với `br-lab` thông qua `veth pair`:

- `veth-ex`: phía `br-ex`
- `veth-lab`: phía `br-lab`

Floating IP range ví dụ: `192.168.150.200/29`

NAT sẽ được cấu hình để dải Floating IP này đi ra ngoài qua `br-lab`.


In [ ]:
# ============================================================
# BƯỚC 10 — Tạo br-ex và NAT Floating IP qua br-lab
# Chạy trên node đóng vai trò Network node.
# Nếu cả 3 server đều chạy network role, cần cân nhắc NAT/route tránh trùng.
# ============================================================

# Tạo bridge br-ex
sudo ip link add br-ex type bridge
sudo ip link set br-ex up

# Tạo veth pair:
# - veth-ex: nối vào br-ex
# - veth-lab: nối vào br-lab
sudo ip link add veth-ex type veth peer name veth-lab

# Gắn veth-ex vào br-ex
sudo ip link set veth-ex master br-ex

# Gắn veth-lab vào br-lab
sudo ip link set veth-lab master br-lab

# Bật hai đầu veth
sudo ip link set veth-ex up
sudo ip link set veth-lab up

# Bật IP forwarding tạm thời
sudo sysctl -w net.ipv4.ip_forward=1

# Bật IP forwarding persistent sau reboot
echo "net.ipv4.ip_forward=1" | sudo tee -a /etc/sysctl.conf

# NAT Floating IP range ra ngoài qua br-lab
# Dải ví dụ: 192.168.150.200/29
sudo iptables -t nat -A POSTROUTING \
  -s 192.168.150.200/29 -o br-lab -j MASQUERADE

# Lưu iptables để còn sau reboot
sudo netfilter-persistent save

# Verify
ip link show br-ex
ip link show veth-ex
ip link show veth-lab
sudo iptables -t nat -S POSTROUTING


## BƯỚC 11 — Cấu hình Persistent sau reboot

Các lệnh `ip link add` chỉ tồn tại tạm thời. Sau reboot sẽ mất.

Vì vậy cần cấu hình persistent gồm 2 phần:

1. Netplan tạo bridge rỗng:
   - `br-mgmt`
   - `br-storage`
   - `br-cluster`
   - `br-ex`

2. Systemd service chạy script tạo VXLAN:
   - `/usr/local/bin/vxlan-setup.sh`
   - `/etc/systemd/system/vxlan-setup.service`

Notebook này viết mẫu cho Server A trước. Khi dùng cho Server B/C cần đổi IP đúng theo bảng IP.


### BƯỚC 11.1 — Netplan persistent cho Server A

In [ ]:
# ============================================================
# BƯỚC 11.1 — SERVER A: tạo file netplan persistent bridge
# File: /etc/netplan/02-openstack-bridges.yaml
# ============================================================

sudo tee /etc/netplan/02-openstack-bridges.yaml << EOF
network:
  version: 2
  renderer: networkd
  bridges:
    br-mgmt:
      interfaces: []
      addresses: [10.0.0.21/24]
      mtu: 1450
      parameters:
        stp: false
        forward-delay: 0

    br-storage:
      interfaces: []
      addresses: [10.0.2.21/24]
      mtu: 1450
      parameters:
        stp: false
        forward-delay: 0

    br-cluster:
      interfaces: []
      addresses: [10.0.3.21/24]
      mtu: 1450
      parameters:
        stp: false
        forward-delay: 0

    br-ex:
      interfaces: []
      parameters:
        stp: false
        forward-delay: 0
EOF

# Kiểm tra cú pháp netplan trước khi apply
sudo netplan generate

# Chỉ apply khi chắc chắn SSH không bị ảnh hưởng
# sudo netplan apply


### BƯỚC 11.2 — Script tạo VXLAN persistent cho Server A

In [ ]:
# ============================================================
# BƯỚC 11.2 — SERVER A: tạo script /usr/local/bin/vxlan-setup.sh
# Script này tạo VXLAN sau reboot.
# Lưu ý: dùng '|| true' để tránh fail nếu interface đã tồn tại.
# ============================================================

sudo tee /usr/local/bin/vxlan-setup.sh << 'EOF'
#!/bin/bash
set -e

# =========================
# br-mgmt — VXLAN id 100
# =========================
ip link add vxlan100-b type vxlan id 100 local 192.168.150.9 remote 192.168.150.10 dstport 4789 nolearning || true
ip link add vxlan100-c type vxlan id 100 local 192.168.150.9 remote 192.168.150.11 dstport 4789 nolearning || true
ip link set vxlan100-b master br-mgmt || true
ip link set vxlan100-c master br-mgmt || true
ip link set vxlan100-b mtu 1450 || true
ip link set vxlan100-c mtu 1450 || true
ip link set vxlan100-b up || true
ip link set vxlan100-c up || true
ip link set br-mgmt up || true

# =========================
# br-storage — VXLAN id 200
# =========================
ip link add vxlan200-b type vxlan id 200 local 192.168.150.9 remote 192.168.150.10 dstport 4789 nolearning || true
ip link add vxlan200-c type vxlan id 200 local 192.168.150.9 remote 192.168.150.11 dstport 4789 nolearning || true
ip link set vxlan200-b master br-storage || true
ip link set vxlan200-c master br-storage || true
ip link set vxlan200-b mtu 1450 || true
ip link set vxlan200-c mtu 1450 || true
ip link set vxlan200-b up || true
ip link set vxlan200-c up || true
ip link set br-storage up || true

# =========================
# br-cluster — VXLAN id 300
# =========================
ip link add vxlan300-b type vxlan id 300 local 192.168.150.9 remote 192.168.150.10 dstport 4789 nolearning || true
ip link add vxlan300-c type vxlan id 300 local 192.168.150.9 remote 192.168.150.11 dstport 4789 nolearning || true
ip link set vxlan300-b master br-cluster || true
ip link set vxlan300-c master br-cluster || true
ip link set vxlan300-b mtu 1450 || true
ip link set vxlan300-c mtu 1450 || true
ip link set vxlan300-b up || true
ip link set vxlan300-c up || true
ip link set br-cluster up || true

# =========================
# br-ex + veth pair
# =========================
ip link add veth-ex type veth peer name veth-lab || true
ip link set veth-ex master br-ex || true
ip link set veth-lab master br-lab || true
ip link set veth-ex up || true
ip link set veth-lab up || true
ip link set br-ex up || true

exit 0
EOF

sudo chmod +x /usr/local/bin/vxlan-setup.sh


### BƯỚC 11.3 — Tạo systemd service chạy VXLAN script

In [ ]:
# ============================================================
# BƯỚC 11.3 — Tạo systemd service cho vxlan-setup.sh
# ============================================================

sudo tee /etc/systemd/system/vxlan-setup.service << EOF
[Unit]
Description=VXLAN Bridge Setup for OpenStack
After=network-online.target
Wants=network-online.target

[Service]
Type=oneshot
ExecStart=/usr/local/bin/vxlan-setup.sh
RemainAfterExit=yes

[Install]
WantedBy=multi-user.target
EOF

# Reload systemd
sudo systemctl daemon-reload

# Enable service sau reboot
sudo systemctl enable vxlan-setup.service

# Chạy thử ngay
sudo systemctl start vxlan-setup.service

# Verify trạng thái service
sudo systemctl status vxlan-setup.service --no-pager


### BƯỚC 11.4 — Verify tổng thể sau persistent config

In [ ]:
# ============================================================
# VERIFY tổng thể network fabric
# ============================================================

# Kiểm tra bridge
ip addr show br-mgmt
ip addr show br-storage
ip addr show br-cluster
ip link show br-ex

# Kiểm tra VXLAN interface
ip link | grep vxlan

# Kiểm tra routing
ip route

# Kiểm tra ping overlay từ Server A
ping -c 3 -I br-mgmt 10.0.0.22
ping -c 3 -I br-mgmt 10.0.0.23

ping -c 3 -I br-storage 10.0.2.22
ping -c 3 -I br-storage 10.0.2.23

ping -c 3 -I br-cluster 10.0.3.22
ping -c 3 -I br-cluster 10.0.3.23

# Kiểm tra NAT rule
sudo iptables -t nat -S POSTROUTING


## Ghi chú debug nhanh

### 1. Không ping được overlay
Kiểm tra theo thứ tự:

```bash
ping -c 3 192.168.150.10
ping -c 3 192.168.150.11
ip link | grep vxlan
ip addr show br-mgmt
sudo ufw status
```

Nếu underlay `192.168.150.x` không ping được thì VXLAN chắc chắn không hoạt động.

### 2. VXLAN bị mất sau reboot
Kiểm tra:

```bash
sudo systemctl status vxlan-setup.service
journalctl -u vxlan-setup.service -xe
```

### 3. MTU mismatch
Kiểm tra:

```bash
ip link show br-mgmt
ip link show br-storage
ip link show br-cluster
```

VXLAN nên dùng MTU `1450`.

### 4. Floating IP không ra ngoài
Kiểm tra:

```bash
sysctl net.ipv4.ip_forward
sudo iptables -t nat -S POSTROUTING
ip link show br-ex
ip link show veth-ex
ip link show veth-lab
```
